# Frozen ResNet, ViT and CLIP wrappers
Reorganized only; no experiment cells executed during creation. Existing results were copied with byte-hash verification.

In [ ]:
def load_backbone(name):
    if name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        backbone = models.resnet50(weights=weights)

        # Removing fc exposes the flattened global-average-pooled feature.
        backbone.fc = nn.Identity()

        preprocessing = weights.transforms()
        mean, std = preprocessing.mean, preprocessing.std

    elif name == "vit_b16":
        weights = models.ViT_B_16_Weights.IMAGENET1K_V1
        backbone = models.vit_b_16(weights=weights)

        # Removing the classifier exposes the final encoded class token.
        backbone.heads = nn.Identity()

        preprocessing = weights.transforms()
        mean, std = preprocessing.mean, preprocessing.std

    elif name == "clip":
        backbone, _, preprocessing = open_clip.create_model_and_transforms(
            "ViT-B-32",
            pretrained="openai",
        )

        # Keep CLIP's normalization, while using our common image geometry.
        normalization = next(
            operation
            for operation in reversed(preprocessing.transforms)
            if isinstance(operation, transforms.Normalize)
        )
        mean, std = normalization.mean, normalization.std

    else:
        raise ValueError(f"Unknown backbone: {name}")

    backbone.requires_grad_(False)
    backbone.eval()
    backbone.to(DEVICE)

    normalize = transforms.Normalize(mean=mean, std=std)
    return backbone, normalize


@torch.no_grad()
def extract_features(backbone, normalize, loader, name, split):
    all_features = []
    all_labels = []

    for images, labels in tqdm(loader, desc=f"{name}: {split}"):
        images = normalize(images.to(DEVICE))

        if name == "clip":
            features = backbone.encode_image(images)
            features = F.normalize(features.float(), dim=-1)
        else:
            features = backbone(images).float()

        all_features.append(features.cpu())
        all_labels.append(labels)

    return {
        "features": torch.cat(all_features),
        "labels": torch.cat(all_labels),
    }

In [ ]:
def train_linear_head(train_data, val_data):
    # Reset the seed independently for every classifier comparison.
    seed_everything(SEED)

    x_train = train_data["features"]
    y_train = train_data["labels"]
    x_val = val_data["features"]
    y_val = val_data["labels"]

    # Training on cached features is inexpensive enough to do on CPU.
    head = nn.Linear(x_train.shape[1], num_classes)

    optimizer = torch.optim.AdamW(
        head.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss()

    generator = torch.Generator().manual_seed(SEED)

    train_loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )

    best_val_accuracy = -1.0
    best_state = None
    best_epoch = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        head.train()
        total_loss = 0.0

        for features, labels in train_loader:
            optimizer.zero_grad(set_to_none=True)

            logits = head(features)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(labels)

        head.eval()

        with torch.no_grad():
            val_predictions = head(x_val).argmax(dim=1)
            val_accuracy = (
                (val_predictions == y_val).float().mean().item()
            )

        history.append({
            "epoch": epoch,
            "train_loss": total_loss / len(y_train),
            "val_accuracy": val_accuracy,
        })

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(head.state_dict())
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    head.load_state_dict(best_state)
    head.eval()

    return head, pd.DataFrame(history), {
        "best_epoch": best_epoch,
        "epochs_run": epoch,
        "best_val_accuracy": best_val_accuracy,
    }

In [ ]:
import matplotlib.pyplot as plt


def show_tensor(ax, image, title):
    image = image.detach().cpu().clamp(0, 1)
    ax.imshow(image.permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=10)
    ax.axis("off")